# Unit 2 — Advanced Visualization & Storytelling
## Tumor Screening Dashboard


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import comb
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
df=pd.read_csv('breast_cancer.csv').drop(columns=['Unnamed: 32'],errors='ignore'); feature_cols=[c for c in df.columns if c not in ['id','diagnosis']]; X_raw=df[feature_cols].values; y=df.diagnosis.values


In [ ]:
p=len(feature_cols); n_pairs=comb(p,2); n_cells=10**p; print(p,n_pairs,f'{n_cells:.2e}')
corr=df[feature_cols].corr(); plt.figure(figsize=(8,7)); plt.imshow(corr,cmap='RdBu_r',vmin=-1,vmax=1); plt.colorbar(); plt.title('Correlogram'); plt.show(); mask=np.triu(np.ones(corr.shape),1).astype(bool); print('Highly correlated pairs:',int((corr.where(mask).abs()>.9).sum().sum()))


In [ ]:
def pca_from_scratch(X,k):
    X_scaled=(X-X.mean(0))/X.std(0,ddof=1); cov=np.cov(X_scaled,rowvar=False); eigvals,eigvecs=np.linalg.eigh(cov); order=np.argsort(eigvals)[::-1]; eigvals=eigvals[order]; eigvecs=eigvecs[:,order]; return X_scaled@eigvecs[:,:k],eigvals,eigvecs
Z,eigvals,eigvecs=pca_from_scratch(X_raw,2); plt.scatter(Z[y=='B',0],Z[y=='B',1],label='B'); plt.scatter(Z[y=='M',0],Z[y=='M',1],label='M'); plt.legend(); plt.title('From-scratch PCA'); plt.show()
explained_ratio=eigvals/eigvals.sum(); cumulative=np.cumsum(explained_ratio); k_90=int(np.argmax(cumulative>=.9)+1); plt.plot(range(1,len(cumulative)+1),cumulative*100,'o-'); plt.axhline(90,ls='--'); plt.title(f'k={k_90} for 90% variance'); plt.show()


In [ ]:
X_scaled=StandardScaler().fit_transform(X_raw); pca_emb=PCA(2).fit_transform(X_scaled); tsne_emb=TSNE(n_components=2,perplexity=30,random_state=0).fit_transform(X_scaled); umap_emb=umap.UMAP(n_neighbors=15,min_dist=.1,random_state=0).fit_transform(X_scaled)
fig,axes=plt.subplots(1,3,figsize=(15,4));
for ax,emb,name in zip(axes,[pca_emb,tsne_emb,umap_emb],['PCA','t-SNE','UMAP']):
    for label,color in [('B','#3b6ea5'),('M','#b5432e')]:
        m=y==label; ax.scatter(emb[m,0],emb[m,1],s=12,alpha=.6,c=color,label=label)
    ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()
